# 02. BPv7 실전: 블록 모델, CRC, fragmentation/reassembly

작성·검증 기준일: **2026-09-11**  
원문: [RFC 9171 — Bundle Protocol Version 7](https://www.rfc-editor.org/rfc/rfc9171.html)

## 학습 목표

1. primary block과 canonical block의 불변 조건을 데이터 모델로 표현한다.
2. X-25 CRC-16과 CRC32C(Castagnoli)를 정확한 check vector로 검증한다.
3. fragmentation 후 순서가 뒤섞인 fragment를 재조립하고, 중복·overlap·범위 오류를 처리한다.

> **범위 경계:** 데이터 클래스와 reassembler는 RFC 규칙을 학습하기 위한 toy이다. 블록을 CBOR wire bytes로 만들지 않으며 실제 BPv7 상호운용 구현이 아니다. CRC 함수 자체는 표준 알고리즘을 구현하지만, 예시 `crc_for_zeroed_field` 입력은 CRC **값 바이트만** 0으로 채운 직렬화 바이트라고 가정한다. 실제 구현은 RFC 9171 §4.3의 CBOR 전체 블록 바이트를 정확히 구성해야 한다.

관련 절: RFC 9171 §4.1–4.3, §5.6, §5.8–5.9. 실행은 **Run All**, 의존성은 Python 표준 라이브러리뿐이다.

## 1. Primary / canonical block 불변 조건

RFC의 핵심 형식 규칙을 모두 담지는 않지만, 오류가 자주 생기는 조건을 코드로 고정한다.

- primary block의 version은 7이고 전송 중 불변이다(여기서는 frozen dataclass).
- primary CRC type 0은 primary block을 대상으로 한 BIB가 있을 때만 가능하다.
- fragment flag와 fragment offset/total ADU length는 함께 나타나야 한다.
- canonical block 번호는 bundle 안에서 유일하며 payload block 번호는 1이다.
- payload block은 정확히 하나이고 마지막 canonical block이다.

In [1]:
from dataclasses import dataclass

FRAGMENT_FLAG = 0x000001
PAYLOAD_BLOCK_TYPE = 1

@dataclass(frozen=True)
class PrimaryBlock:
    destination: str
    source: str
    report_to: str
    creation_time_ms: int
    sequence_number: int
    lifetime_ms: int
    flags: int = 0
    crc_type: int = 1
    fragment_offset: int | None = None
    total_adu_length: int | None = None
    primary_bib_protected: bool = False
    version: int = 7

    def validate(self, payload_length: int) -> None:
        integer_fields = (self.version, self.flags, self.crc_type, self.creation_time_ms,
                          self.sequence_number, self.lifetime_ms, payload_length)
        if any(type(value) is not int or value < 0 for value in integer_fields):
            raise ValueError("정수 필드는 음이 아닌 int여야 합니다")
        if self.version != 7:
            raise ValueError("BPv7 primary version은 7이어야 합니다")
        if self.crc_type not in (0, 1, 2):
            raise ValueError("CRC type은 0, 1, 2 중 하나여야 합니다")
        if self.crc_type == 0 and not self.primary_bib_protected:
            raise ValueError("primary CRC 생략에는 primary-targeted BIB가 필요합니다")
        is_fragment = bool(self.flags & FRAGMENT_FLAG)
        has_fragment_fields = self.fragment_offset is not None and self.total_adu_length is not None
        if is_fragment != has_fragment_fields:
            raise ValueError("fragment flag와 offset/total length의 존재가 일치해야 합니다")
        if is_fragment:
            if type(self.fragment_offset) is not int or type(self.total_adu_length) is not int:
                raise ValueError("fragment 필드는 int여야 합니다")
            if self.fragment_offset < 0 or self.total_adu_length <= 0:
                raise ValueError("fragment 범위가 잘못되었습니다")
            if self.fragment_offset + payload_length > self.total_adu_length:
                raise ValueError("fragment가 원 ADU 범위를 벗어났습니다")

@dataclass(frozen=True)
class CanonicalBlock:
    type_code: int
    block_number: int
    data: bytes
    flags: int = 0
    crc_type: int = 1

    def validate(self) -> None:
        if any(type(value) is not int or value < 0
               for value in (self.type_code, self.block_number, self.flags, self.crc_type)):
            raise ValueError("canonical block 숫자 필드가 잘못되었습니다")
        if self.block_number == 0:
            raise ValueError("0은 primary block의 암시적 번호입니다")
        if self.crc_type not in (0, 1, 2):
            raise ValueError("CRC type은 0, 1, 2 중 하나여야 합니다")
        if not isinstance(self.data, bytes):
            raise TypeError("block data는 bytes여야 합니다")

@dataclass(frozen=True)
class BundleModel:
    primary: PrimaryBlock
    blocks: tuple[CanonicalBlock, ...]

    def validate(self) -> None:
        if not self.blocks:
            raise ValueError("primary 뒤에 canonical block이 하나 이상 필요합니다")
        for block in self.blocks:
            block.validate()
        numbers = [block.block_number for block in self.blocks]
        if len(numbers) != len(set(numbers)):
            raise ValueError("canonical block 번호는 bundle 안에서 유일해야 합니다")
        payloads = [block for block in self.blocks if block.type_code == PAYLOAD_BLOCK_TYPE]
        if len(payloads) != 1 or payloads[0].block_number != 1:
            raise ValueError("payload block은 정확히 하나이며 번호가 1이어야 합니다")
        if self.blocks[-1] is not payloads[0]:
            raise ValueError("payload block은 마지막 canonical block이어야 합니다")
        self.primary.validate(len(payloads[0].data))

bundle = BundleModel(
    PrimaryBlock("dtn://mars/inbox", "dtn://earth", "dtn://earth/report",
                 creation_time_ms=42, sequence_number=0, lifetime_ms=60_000),
    (CanonicalBlock(type_code=7, block_number=2, data=b"\x19\x03\xe8"),
     CanonicalBlock(type_code=1, block_number=1, data=b"science payload")),
)
bundle.validate()
print("유효한 toy bundle:", bundle)

유효한 toy bundle: BundleModel(primary=PrimaryBlock(destination='dtn://mars/inbox', source='dtn://earth', report_to='dtn://earth/report', creation_time_ms=42, sequence_number=0, lifetime_ms=60000, flags=0, crc_type=1, fragment_offset=None, total_adu_length=None, primary_bib_protected=False, version=7), blocks=(CanonicalBlock(type_code=7, block_number=2, data=b'\x19\x03\xe8', flags=0, crc_type=1), CanonicalBlock(type_code=1, block_number=1, data=b'science payload', flags=0, crc_type=1)))


## 2. CRC-16/X-25와 CRC32C

RFC 9171의 CRC type 1은 X-25 CRC-16, type 2는 CRC32C(Castagnoli)이다. CRC는 우발적 손상 검출용이지 공격자에 대한 인증이 아니다. BPSec의 BIB와 혼동하지 않는다.

[Verified Errata 8043](https://www.rfc-editor.org/errata/eid8043)은 CRC 계산 전에 **CBOR byte-string header(초기 바이트)는 그대로 유지하고 CRC field의 value bytes만 0으로 만든다**고 명확히 한다. 예를 들어 CRC-16 byte string의 header `0x42`는 0으로 바꾸지 않고 그 뒤 두 값 바이트만 `00 00`으로 둔다.

아래 구현은 reflected polynomial을 사용한다. 잘 알려진 ASCII `123456789` check 값은 각각 `0x906e`, `0xe3069283`이다.

In [2]:
def crc16_x25(data: bytes) -> int:
    """CRC-16/IBM-SDLC(X-25): poly=0x1021, refin/out, init/xorout=0xffff."""
    crc = 0xFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0x8408 if crc & 1 else crc >> 1
    return crc ^ 0xFFFF

def crc32c(data: bytes) -> int:
    """CRC-32/ISCSI(Castagnoli): reflected poly=0x82f63b78."""
    crc = 0xFFFFFFFF
    for byte in data:
        crc ^= byte
        for _ in range(8):
            crc = (crc >> 1) ^ 0x82F63B78 if crc & 1 else crc >> 1
    return crc ^ 0xFFFFFFFF

CHECK = b"123456789"
assert crc16_x25(CHECK) == 0x906E
assert crc32c(CHECK) == 0xE3069283
print(f"CRC-16/X-25: {crc16_x25(CHECK):04x}")
print(f"CRC32C:        {crc32c(CHECK):08x}")

CRC-16/X-25: 906e
CRC32C:        e3069283


In [3]:
def crc_for_zeroed_field(serialized_block_with_zero_crc: bytes, crc_type: int) -> bytes:
    """CBOR header는 유지하고 CRC value bytes만 0인 전체 블록을 받는다(Errata 8043)."""
    if crc_type == 1:
        return crc16_x25(serialized_block_with_zero_crc).to_bytes(2, "big")
    if crc_type == 2:
        return crc32c(serialized_block_with_zero_crc).to_bytes(4, "big")
    if crc_type == 0:
        return b""
    raise ValueError("알 수 없는 CRC type")

# 실제 BP 블록이 아닌 synthetic sequence다. 실제 CBOR의 byte-string header는 0으로 만들면 안 된다.
toy_serialization = b"canonical-block-body" + bytes(2)
attached = crc_for_zeroed_field(toy_serialization, 1)
damaged = bytearray(toy_serialization)
damaged[2] ^= 1
assert crc_for_zeroed_field(bytes(damaged), 1) != attached
print("network-byte-order CRC bytes:", attached.hex())
print("1-bit 손상 검출:", crc_for_zeroed_field(bytes(damaged), 1) != attached)

network-byte-order CRC bytes: 9fe6
1-bit 손상 검출: True


## 3. Fragmentation과 material extents 재조립

RFC 9171 §5.8은 payload를 연속 구간으로 나누며, §5.9는 같은 source node ID와 creation timestamp를 가진 fragment의 **새로운 material extents**를 모아 전체 ADU를 만든다. 서로 다른 fragmentation episode에서 overlap이 생길 수 있다.

아래 reassembler 정책은 다음과 같다.

- 동일 범위·동일 바이트 재전송은 `duplicate`로 멱등 처리한다.
- 일부 overlap이 있어도 기존 바이트와 같으면 새 material byte만 반영한다.
- overlap 바이트가 다르거나 범위를 벗어나면 거부한다. RFC가 충돌 해결 알고리즘을 정하지 않으므로 이는 안전한 로컬 정책이다.
- 선언된 ADU 크기와 fragment 개수에 상한을 둔다. 이는 실제 구현에 필요한 자원 방어다.

In [4]:
@dataclass(frozen=True)
class Fragment:
    source: str
    creation_timestamp: tuple[int, int]
    offset: int
    total_length: int
    payload: bytes

    @property
    def key(self):
        return self.source, self.creation_timestamp

    def validate(self) -> None:
        if type(self.offset) is not int or type(self.total_length) is not int:
            raise ValueError("offset과 total_length는 int여야 합니다")
        if self.offset < 0 or self.total_length <= 1:
            raise ValueError("fragment 범위가 잘못되었습니다")
        if not isinstance(self.payload, bytes) or not self.payload:
            raise ValueError("fragment payload는 비어 있지 않은 bytes여야 합니다")
        if len(self.payload) >= self.total_length:
            raise ValueError("fragment는 원 ADU보다 작아야 합니다")
        if self.offset + len(self.payload) > self.total_length:
            raise ValueError("fragment가 ADU 범위를 벗어났습니다")

def fragment_payload(source: str, timestamp: tuple[int, int], payload: bytes,
                     max_payload: int) -> list[Fragment]:
    if not isinstance(payload, bytes) or len(payload) < 2:
        raise ValueError("두 바이트 이상의 bytes payload가 필요합니다")
    if type(max_payload) is not int or not 0 < max_payload < len(payload):
        raise ValueError("max_payload는 0보다 크고 payload보다 작아야 합니다")
    return [Fragment(source, timestamp, offset, len(payload), payload[offset:offset + max_payload])
            for offset in range(0, len(payload), max_payload)]

class Reassembler:
    def __init__(self, key, total_length: int, *, max_total_length=1 << 20, max_fragments=4096):
        if not 0 < total_length <= max_total_length:
            raise ValueError("선언된 ADU 크기가 로컬 한도를 벗어났습니다")
        self.key = key
        self.total_length = total_length
        self.max_fragments = max_fragments
        self.data = bytearray(total_length)
        self.present = bytearray(total_length)
        self.signatures: dict[tuple[int, int], bytes] = {}
        self.unique_fragments = 0
        self.material_bytes = 0

    def add(self, fragment: Fragment) -> str:
        fragment.validate()
        if fragment.key != self.key or fragment.total_length != self.total_length:
            raise ValueError("다른 bundle identity 또는 total length")
        span = (fragment.offset, len(fragment.payload))
        if span in self.signatures:
            if self.signatures[span] != fragment.payload:
                raise ValueError("같은 범위의 충돌하는 duplicate")
            return "duplicate"
        if self.unique_fragments >= self.max_fragments:
            raise MemoryError("fragment 개수 한도 초과")
        for index, byte in enumerate(fragment.payload, start=fragment.offset):
            if self.present[index] and self.data[index] != byte:
                raise ValueError(f"offset {index}에서 overlap 내용 충돌")
        new_bytes = 0
        for index, byte in enumerate(fragment.payload, start=fragment.offset):
            if not self.present[index]:
                self.data[index] = byte
                self.present[index] = 1
                new_bytes += 1
        self.signatures[span] = fragment.payload
        self.unique_fragments += 1
        self.material_bytes += new_bytes
        return "complete" if self.complete else ("overlap" if new_bytes < len(fragment.payload) else "added")

    @property
    def complete(self) -> bool:
        return self.material_bytes == self.total_length

    def result(self) -> bytes:
        if not self.complete:
            raise RuntimeError("아직 연속된 전체 ADU가 모이지 않았습니다")
        return bytes(self.data)

In [5]:
adu = b"Delay tolerant payload across intermittent links"
fragments = fragment_payload("dtn://earth", (1234, 0), adu, max_payload=9)
assembler = Reassembler(fragments[0].key, len(adu))
order = list(reversed(range(len(fragments))))
for index in order:
    status = assembler.add(fragments[index])
    print(f"offset={fragments[index].offset:2}: {status:8} material={assembler.material_bytes}")
assert assembler.result() == adu
assert assembler.add(fragments[0]) == "duplicate"
print("재조립 결과:", assembler.result().decode())

offset=45: added    material=3
offset=36: added    material=12
offset=27: added    material=21
offset=18: added    material=30
offset= 9: added    material=39
offset= 0: complete material=48
재조립 결과: Delay tolerant payload across intermittent links


In [6]:
def expect_error(label, function, error_type):
    try:
        function()
    except error_type as exc:
        print(f"{label}: {type(exc).__name__} - {exc}")
    else:
        raise AssertionError(f"{label}은(는) 오류여야 합니다")

key = ("dtn://earth", (1234, 0))
overlap_assembler = Reassembler(key, len(adu))
first = Fragment(key[0], key[1], 0, len(adu), adu[:9])
consistent_overlap = Fragment(key[0], key[1], 7, len(adu), adu[7:14])
assert overlap_assembler.add(first) == "added"
assert overlap_assembler.add(consistent_overlap) == "overlap"

expect_error(
    "충돌 overlap",
    lambda: overlap_assembler.add(Fragment(key[0], key[1], 8, len(adu), b"Xbad")),
    ValueError,
)
expect_error(
    "범위 초과",
    lambda: overlap_assembler.add(Fragment(key[0], key[1], len(adu) - 2, len(adu), b"toolong")),
    ValueError,
)
expect_error(
    "identity 혼합",
    lambda: overlap_assembler.add(Fragment("dtn://other", key[1], 9, len(adu), adu[9:12])),
    ValueError,
)

충돌 overlap: ValueError - offset 8에서 overlap 내용 충돌
범위 초과: ValueError - fragment가 ADU 범위를 벗어났습니다
identity 혼합: ValueError - 다른 bundle identity 또는 total length


## 4. 자동 검수와 예상 결과

아래 셀은 모델 불변 조건, CRC check vector, shuffled reassembly, duplicate 처리, 자원 상한을 다시 검증한다. 성공 시 `응용 실습 검수 통과`가 출력된다.

In [7]:
bundle.validate()
assert crc16_x25(b"123456789") == 0x906E
assert crc32c(b"123456789") == 0xE3069283
assert assembler.complete and assembler.result() == adu

bad_payload_number = BundleModel(
    bundle.primary, (CanonicalBlock(1, 2, b"payload"),)
)
expect_error("payload block number", bad_payload_number.validate, ValueError)
expect_error("과대 ADU 선언", lambda: Reassembler(key, 100, max_total_length=10), ValueError)

one_slot = Reassembler(key, len(adu), max_fragments=1)
one_slot.add(first)
expect_error("fragment 개수 한도", lambda: one_slot.add(consistent_overlap), MemoryError)
print("응용 실습 검수 통과")

payload block number: ValueError - payload block은 정확히 하나이며 번호가 1이어야 합니다
과대 ADU 선언: ValueError - 선언된 ADU 크기가 로컬 한도를 벗어났습니다
fragment 개수 한도: MemoryError - fragment 개수 한도 초과
응용 실습 검수 통과


## 연습 문제

1. `BundleModel.validate()`에 Previous Node(type 6), Bundle Age(type 7), Hop Count(type 10)가 각각 최대 한 번만 나타나는 검사를 추가하라.
2. fragment가 도착할 때마다 `present`의 연속 구간 목록을 출력하라.
3. CRC 필드를 0으로 둔 CBOR canonical block을 01의 encoder로 구성하라. 단, 그 encoder가 BPv7 전체 wire format을 지원하지 않는 이유도 적어라.
4. overlap 충돌 시 첫 값 유지, 전체 bundle 폐기, BPSec 검증 대기 정책의 장단점을 비교하라.

다음 노트북은 수신·dispatch·forward·deliver·delete 상태와 공격 표면을 하나의 bounded simulator로 묶는다.